In [91]:
from transformer_scratch import build_transformer
from torch.nn.utils.rnn import pad_sequence
import torch


In [92]:
import sentencepiece as spm

In [93]:
eng_tokenizer = spm.SentencePieceProcessor()
eng_tokenizer.load('english_tokenizer.model')

True

In [94]:
hindi_tokenizer = spm.SentencePieceProcessor()
hindi_tokenizer.load('hindi_tokenizer.model')

True

In [95]:
src_vocab_size = eng_tokenizer.get_piece_size()
tgt_vocab_size = hindi_tokenizer.get_piece_size()

In [96]:
max_src_length = 61
tgt_length = 61
seq_length = 61

In [97]:
model = build_transformer(
    8001,
    8001,
    max_src_length,
    tgt_length,
    256,   # d_model
    4,     # layers
    8,     # heads
    0.1,
    1024
)

In [98]:
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn

In [99]:
class LangDataset:
    def __init__(self, df, src_tokenize, tgt_tokenize, max_seq_length):
        self.df = df
        self.max_seq_length = max_seq_length

        eng_sentences = df['English'].astype(str).tolist()
        hi_sentences = df['Hindi'].astype(str).tolist()

        self.eng_tokenized = src_tokenize.encode(
            eng_sentences,
            out_type=int,
            add_eos = True,
            add_bos = True
        )

        self.hi_tokenized = tgt_tokenize.encode(
            hi_sentences,
            out_type=int,
            add_eos = True,
            add_bos = True
        )


    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):
        eng_token = self.eng_tokenized[idx]
        if len(eng_token) > 61:
            eng_token = eng_token[:61]
            eng_token[-1] = 2
        else:
            eng_token = eng_token + [8000] * (self.max_seq_length - (len(eng_token)))
            
        hi_token =self.hi_tokenized[idx]
        if len(hi_token) > 61:
            hi_token = hi_token[:61]
            hi_token[-1] = 2
        else:
            hi_token = hi_token + [8000] * (self.max_seq_length - (len(hi_token)))





        return torch.tensor(eng_token) , torch.tensor(hi_token)


In [100]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("preetviradiya/english-hindi-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/legion/.cache/kagglehub/datasets/preetviradiya/english-hindi-dataset/versions/1


In [101]:
df_path = path + '/Dataset_English_Hindi.csv'

In [102]:
import pandas as pd
df = pd.read_csv(df_path)

In [103]:
dataset = LangDataset(df.dropna(), eng_tokenizer, hindi_tokenizer, 61)

In [104]:
dataloader = DataLoader(dataset, 78, shuffle = True)

In [ ]:
lossfn = nn.CrossEntropyLoss(ignore_index = 8000)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [106]:
from tqdm import tqdm

In [107]:
epochs = 7

In [108]:
device = torch.device('cuda')

In [109]:
model = model.to(device)

In [110]:
for i in range(epochs):
    total = 0

    for eng_token, hi_token in tqdm(
        dataloader,
        desc="Training English -> Hindi Translation Model",
        unit="batch"
    ):

        eng_token = eng_token.to(device)
        hi_token = hi_token.to(device)

        src_mask = (eng_token != 8000).unsqueeze(1).unsqueeze(2)

        decoder_input = hi_token[:, :-1]
        targets = hi_token[:, 1:]

        seq_len = decoder_input.shape[1]

        causal_mask = torch.tril(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
        ).unsqueeze(0).unsqueeze(0)

        pad_mask = (decoder_input != 8000).unsqueeze(1).unsqueeze(2)

        tgt_mask = causal_mask & pad_mask

        encoder_output = model.encode(eng_token, src_mask)

        decoder_output = model.decode(
            encoder_output,
            src_mask,
            decoder_input,
            tgt_mask
        )

        logits = model.project(decoder_output)

        optimizer.zero_grad(set_to_none=True)

        loss = lossfn(
            logits.reshape(-1, 8001),
            targets.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total += loss.item()

    avg_loss = total / len(dataloader)



    print(f"Epoch {i+1} Loss: {avg_loss:.4f}")

Training English -> Hindi Translation Model: 100%|██████████| 1669/1669 [02:56<00:00,  9.45batch/s]


Epoch 1 Loss: 5.5051


Training English -> Hindi Translation Model:   1%|▏         | 25/1669 [00:02<02:58,  9.23batch/s]


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "model.pt")